# 🌿 Plant Disease Dataset Explorer

| Section | What you'll see |
|---|---|
| **1. Overview** | Total images, classes, split sizes |
| **2. Class Distribution** | Per-class image counts — sorted bar chart |
| **3. Imbalance Analysis** | Rare classes (< 50 imgs), weight histogram |
| **4. Split Comparison** | Train / Val / Test counts per class |
| **5. Crop Breakdown** | Which crops dominate the dataset |
| **6. Source Datasets** | Where each image came from |
| **7. Image Dimensions** | Width/height distribution |
| **8. Sample Images** | Visual grid per class |
| **9. TTA Report** | Standard vs TTA F1 per class |
| **10. Search** | Look up any class by keyword |

In [ ]:
import os, json, random, sys
from pathlib import Path
from collections import Counter
from math import ceil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

ROOT       = Path('..')
TRAIN_DIR  = ROOT / 'data' / 'processed' / 'train'
VAL_DIR    = ROOT / 'data' / 'processed' / 'val'
TEST_DIR   = ROOT / 'data' / 'processed' / 'test'
TRAIN_CSV  = ROOT / 'data' / 'raw' / 'outputs' / 'outputs' / 'train_split.csv'
VAL_CSV    = ROOT / 'data' / 'raw' / 'outputs' / 'outputs' / 'val_split.csv'
TEST_CSV   = ROOT / 'data' / 'raw' / 'outputs' / 'outputs' / 'test_split.csv'
STD_REPORT = ROOT / 'models' / 'classification_report_standard.json'
TTA_REPORT = ROOT / 'models' / 'classification_report_tta.json'

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3a3d4d', 'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#a0a0b0', 'ytick.color': '#a0a0b0',
    'text.color': '#e0e0e0', 'grid.color': '#2a2d3a',
    'grid.linestyle': '--', 'grid.alpha': 0.6, 'font.size': 11,
})
ACCENT, ACCENT2, DANGER, WARN = '#4ade80', '#60a5fa', '#f87171', '#fbbf24'
print('✅ Setup done')

## 1 · Dataset Overview

In [ ]:
def count_images_per_class(split_dir):
    counts = {}
    for d in sorted(Path(split_dir).iterdir()):
        if d.is_dir():
            counts[d.name] = sum(
                1 for f in d.rglob('*')
                if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}
            )
    return counts

train_counts = count_images_per_class(TRAIN_DIR)
val_counts   = count_images_per_class(VAL_DIR)
test_counts  = count_images_per_class(TEST_DIR)
all_classes  = sorted(train_counts.keys())

n_classes = len(train_counts)
n_train   = sum(train_counts.values())
n_val     = sum(val_counts.values())
n_test    = sum(test_counts.values())
n_total   = n_train + n_val + n_test
tv = list(train_counts.values())

print('=' * 52)
print(f'  🌿 Total classes          : {n_classes:>6,}')
print(f'  🖼️  Total images           : {n_total:>6,}')
print('-' * 52)
print(f'  📦 Train images           : {n_train:>6,}  ({n_train/n_total*100:.1f}%)')
print(f'  📦 Val images             : {n_val:>6,}  ({n_val/n_total*100:.1f}%)')
print(f'  📦 Test images            : {n_test:>6,}  ({n_test/n_total*100:.1f}%)')
print('-' * 52)
print(f'  📊 Train min / class      : {min(tv):>6,}')
print(f'  📊 Train max / class      : {max(tv):>6,}')
print(f'  📊 Train mean / class     : {np.mean(tv):>9.1f}')
print(f'  📊 Train median / class   : {np.median(tv):>9.1f}')
print(f'  📊 Imbalance ratio        : {max(tv)/min(tv):>9.1f}x')
print('=' * 52)

## 2 · Class Distribution (Training Set)

In [ ]:
sorted_items  = sorted(train_counts.items(), key=lambda x: x[1], reverse=True)
labels_sorted = [c for c,_ in sorted_items]
counts_sorted = [v for _,v in sorted_items]

bar_colors = [ACCENT if v>=300 else ACCENT2 if v>=100 else WARN if v>=50 else DANGER
              for v in counts_sorted]

fig, ax = plt.subplots(figsize=(22, 6))
ax.bar(range(len(labels_sorted)), counts_sorted, color=bar_colors, width=0.85)
ax.set_xticks([])
ax.set_xlabel('Classes sorted by image count (high to low)', labelpad=8)
ax.set_ylabel('Image count')
ax.set_title('Training Set — Per-Class Image Count', fontsize=14, pad=12)
ax.axhline(np.mean(counts_sorted), color=WARN, linestyle='--', lw=1.5,
           label=f'Mean = {np.mean(counts_sorted):.0f}')
ax.axhline(50, color=DANGER, linestyle=':', lw=1.5, label='Rare threshold (50)')
ax.grid(axis='y')
patches = [
    mpatches.Patch(color=ACCENT,  label='>=300 images'),
    mpatches.Patch(color=ACCENT2, label='100-299 images'),
    mpatches.Patch(color=WARN,    label='50-99 images'),
    mpatches.Patch(color=DANGER,  label='<50 images (RARE)'),
]
ax.legend(handles=patches, loc='upper right', framealpha=0.3)
plt.tight_layout()
plt.show()
print(f'Rare classes (< 50 imgs): {sum(1 for v in counts_sorted if v < 50)}')

## 3 · Imbalance Analysis

In [ ]:
RARE_THRESHOLD = 50
rare = [(cls, cnt) for cls, cnt in sorted(train_counts.items(), key=lambda x: x[1])
        if cnt < RARE_THRESHOLD]
print(f'Classes with < {RARE_THRESHOLD} training images: {len(rare)}')
df_rare = pd.DataFrame(rare, columns=['Class', 'Train'])
df_rare['Val']   = df_rare['Class'].map(val_counts)
df_rare['Test']  = df_rare['Class'].map(test_counts)
df_rare['Total'] = df_rare['Train'] + df_rare['Val'] + df_rare['Test']
print(df_rare.to_string(index=False))

In [ ]:
if rare:
    top_rare = df_rare.head(40)
    fig, ax = plt.subplots(figsize=(12, max(6, len(top_rare)*0.38)))
    colors = [DANGER if v < 20 else WARN for v in top_rare['Train']]
    bars = ax.barh(top_rare['Class'], top_rare['Train'], color=colors, height=0.72)
    ax.set_xlabel('Training image count')
    ax.set_title(f'Rarest Classes (< {RARE_THRESHOLD} train images)', fontsize=13, pad=10)
    ax.axvline(RARE_THRESHOLD, color=WARN, linestyle='--', lw=1.5,
               label=f'Threshold = {RARE_THRESHOLD}')
    ax.legend(framealpha=0.3)
    ax.grid(axis='x')
    for bar in bars:
        ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                str(int(bar.get_width())), va='center', fontsize=8.5, color='#e0e0e0')
    plt.tight_layout()
    plt.show()

In [ ]:
counts_arr = np.array([train_counts[c] for c in sorted(train_counts)])
weights    = n_train / (n_classes * counts_arr)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].hist(list(train_counts.values()), bins=40, color=ACCENT2,
             edgecolor='#0f1117', linewidth=0.5)
axes[0].set_xlabel('Images per class')
axes[0].set_ylabel('Number of classes')
axes[0].set_title('Class Size Distribution')
axes[0].axvline(np.mean(list(train_counts.values())), color=WARN,
                linestyle='--', lw=1.5, label='Mean')
axes[0].legend(framealpha=0.3)
axes[0].grid(axis='y')

axes[1].hist(weights, bins=40, color=DANGER, edgecolor='#0f1117', linewidth=0.5)
axes[1].set_xlabel('Class weight (balanced)')
axes[1].set_ylabel('Number of classes')
axes[1].set_title('Balanced Class Weights (used in CrossEntropyLoss)')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 4 · Train / Val / Test Split Comparison

In [ ]:
df_splits = pd.DataFrame({
    'Class': all_classes,
    'Train': [train_counts.get(c,0) for c in all_classes],
    'Val':   [val_counts.get(c,0)   for c in all_classes],
    'Test':  [test_counts.get(c,0)  for c in all_classes],
})
df_splits['Total'] = df_splits[['Train','Val','Test']].sum(axis=1)
df_splits = df_splits.sort_values('Total', ascending=False)

top60 = df_splits.head(60)
fig, ax = plt.subplots(figsize=(22, 6))
x = range(len(top60))
ax.bar(x, top60['Train'], label='Train', color=ACCENT,  width=0.8)
ax.bar(x, top60['Val'],   label='Val',   color=ACCENT2, width=0.8, bottom=top60['Train'])
ax.bar(x, top60['Test'],  label='Test',  color=WARN,    width=0.8,
       bottom=top60['Train'].values + top60['Val'].values)
ax.set_xticks(list(x))
ax.set_xticklabels(top60['Class'].tolist(), rotation=90, fontsize=7)
ax.set_ylabel('Image count')
ax.set_title('Top 60 Classes — Train / Val / Test', fontsize=13, pad=10)
ax.legend(framealpha=0.3)
ax.grid(axis='y')
plt.tight_layout()
plt.show()
print(df_splits[['Class','Train','Val','Test','Total']].head(20).to_string(index=False))

## 5 · Crop Breakdown

In [ ]:
if TRAIN_CSV.exists():
    df_train    = pd.read_csv(TRAIN_CSV)
    crop_counts = df_train['crop'].value_counts()
else:
    crops = Counter()
    for cls in all_classes:
        crop = cls.split(' - ')[0].strip() if ' - ' in cls else cls
        crops[crop] += train_counts.get(cls, 0)
    crop_counts = pd.Series(dict(crops)).sort_values(ascending=False)
    df_train = None

colors_crop = sns.color_palette('husl', len(crop_counts))
fig, axes   = plt.subplots(1, 2, figsize=(18, 5))

axes[0].barh(crop_counts.index, crop_counts.values, color=colors_crop, height=0.75)
axes[0].set_xlabel('Training images')
axes[0].set_title('Images per Crop', pad=8)
axes[0].grid(axis='x')
for i, v in enumerate(crop_counts.values):
    axes[0].text(v+10, i, str(v), va='center', fontsize=8.5, color='#e0e0e0')

axes[1].pie(crop_counts.values, labels=crop_counts.index,
            autopct='%1.1f%%', colors=colors_crop, startangle=140,
            pctdistance=0.75, textprops={'color':'#e0e0e0','fontsize':8})
axes[1].set_title('Proportion by Crop', pad=8)
axes[1].set_facecolor('#1a1d27')

plt.suptitle('Crop Distribution (Training Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6 · Source Dataset Breakdown

In [ ]:
if TRAIN_CSV.exists() and df_train is not None:
    src_counts  = df_train['source_dataset'].value_counts()
    colors_src  = sns.color_palette('mako', len(src_counts))
    fig, axes   = plt.subplots(1, 2, figsize=(18, 5))

    axes[0].barh(src_counts.index, src_counts.values, color=colors_src, height=0.75)
    axes[0].set_xlabel('Training images')
    axes[0].set_title('Images per Source Dataset', pad=8)
    axes[0].grid(axis='x')
    for i, v in enumerate(src_counts.values):
        axes[0].text(v+10, i, str(v), va='center', fontsize=8.5, color='#e0e0e0')

    axes[1].pie(src_counts.values, labels=src_counts.index,
                autopct='%1.1f%%', colors=colors_src, startangle=140,
                pctdistance=0.75, textprops={'color':'#e0e0e0','fontsize':8})
    axes[1].set_title('Proportion by Source Dataset', pad=8)
    axes[1].set_facecolor('#1a1d27')

    plt.suptitle('Source Dataset Distribution', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    print(src_counts.to_string())
else:
    print('train_split.csv not found — skipping source breakdown.')

## 7 · Image Dimensions

In [ ]:
if TRAIN_CSV.exists() and df_train is not None:
    widths  = df_train['width'].dropna()
    heights = df_train['height'].dropna()
else:
    all_imgs = [f for f in TRAIN_DIR.rglob('*')
                if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
    sample   = random.sample(all_imgs, min(500, len(all_imgs)))
    dims     = [Image.open(f).size for f in sample]
    widths   = pd.Series([d[0] for d in dims])
    heights  = pd.Series([d[1] for d in dims])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(widths,  bins=40, color=ACCENT,  edgecolor='#0f1117', linewidth=0.4)
axes[0].set(xlabel='Width (px)', ylabel='Count', title='Width Distribution')
axes[0].grid(axis='y')

axes[1].hist(heights, bins=40, color=ACCENT2, edgecolor='#0f1117', linewidth=0.4)
axes[1].set(xlabel='Height (px)', ylabel='Count', title='Height Distribution')
axes[1].grid(axis='y')

axes[2].scatter(widths, heights, alpha=0.2, s=8, color=WARN)
axes[2].set(xlabel='Width (px)', ylabel='Height (px)', title='Width vs Height')
axes[2].grid()

plt.suptitle('Image Dimensions (Training Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(f'Width  — min:{widths.min():.0f}  max:{widths.max():.0f}  mean:{widths.mean():.0f}  median:{widths.median():.0f}')
print(f'Height — min:{heights.min():.0f}  max:{heights.max():.0f}  mean:{heights.mean():.0f}  median:{heights.median():.0f}')

## 8 · Sample Images per Class

Set `SAMPLE_CLASSES` to a list of class names, or leave as `None` to pick random classes.

In [ ]:
N_CLASSES_TO_SHOW  = 10
N_IMAGES_PER_CLASS = 5
SAMPLE_CLASSES     = None   # e.g. ['Tomato - Late Blight', 'Apple - Scab']

if SAMPLE_CLASSES is None:
    SAMPLE_CLASSES = random.sample(all_classes, min(N_CLASSES_TO_SHOW, len(all_classes)))

fig, axes = plt.subplots(len(SAMPLE_CLASSES), N_IMAGES_PER_CLASS,
                          figsize=(N_IMAGES_PER_CLASS * 2.5, len(SAMPLE_CLASSES) * 2.5))
if len(SAMPLE_CLASSES) == 1:
    axes = [axes]

for ri, cls_name in enumerate(SAMPLE_CLASSES):
    cls_dir   = TRAIN_DIR / cls_name
    imgs      = [f for f in cls_dir.rglob('*')
                 if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
    chosen    = random.sample(imgs, min(N_IMAGES_PER_CLASS, len(imgs)))
    n_imgs    = train_counts.get(cls_name, 0)
    for ci in range(N_IMAGES_PER_CLASS):
        ax = axes[ri][ci]
        if ci < len(chosen):
            ax.imshow(Image.open(chosen[ci]).convert('RGB'))
        ax.axis('off')
        if ci == 0:
            short = cls_name[:30] + '...' if len(cls_name) > 30 else cls_name
            ax.set_ylabel(f'{short}\n({n_imgs})', fontsize=8,
                          rotation=0, ha='right', va='center', labelpad=4)

plt.suptitle('Sample Training Images', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 8b · Drill-down — inspect any single class in detail

In [ ]:
INSPECT_CLASS = all_classes[0]   # change to any class name e.g. 'Tomato - Late Blight'

cls_dir   = TRAIN_DIR / INSPECT_CLASS
img_files = sorted([f for f in cls_dir.rglob('*')
                    if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}])

print(f'Class      : {INSPECT_CLASS}')
print(f'Train imgs : {len(img_files)}')
print(f'Val imgs   : {val_counts.get(INSPECT_CLASS, 0)}')
print(f'Test imgs  : {test_counts.get(INSPECT_CLASS, 0)}')

show_n  = min(20, len(img_files))
chosen  = random.sample(img_files, show_n)
cols, rows = 5, ceil(show_n / 5)

fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
for i, ax in enumerate(axes.flatten()):
    if i < show_n:
        img = Image.open(chosen[i]).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'{img.width}x{img.height}', fontsize=7)
    ax.axis('off')

plt.suptitle(f'{INSPECT_CLASS}  ({len(img_files)} train images)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 9 · TTA Evaluation Report — Standard vs TTA (7 passes)

Run `python -m src.evaluation.evaluate` to generate these reports first.

In [ ]:
if not STD_REPORT.exists() or not TTA_REPORT.exists():
    print('Reports not found. Run:  python -m src.evaluation.evaluate')
else:
    with open(STD_REPORT) as f: std = json.load(f)
    with open(TTA_REPORT) as f: tta = json.load(f)

    print('=' * 55)
    print(f'  Standard accuracy : {std["accuracy"]*100:6.2f}%')
    print(f'  TTA accuracy      : {tta["accuracy"]*100:6.2f}%')
    print(f'  TTA gain          : +{(tta["accuracy"]-std["accuracy"])*100:.2f}%')
    print('-' * 55)
    for key in ['macro avg', 'weighted avg']:
        print(f'  {key}:')
        for m in ['precision','recall','f1-score']:
            s, t = std[key][m], tta[key][m]
            print(f'    {m:<12}: {s:.4f} -> {t:.4f}  ({t-s:+.4f})')
    print('=' * 55)

In [ ]:
if STD_REPORT.exists() and TTA_REPORT.exists():
    skip = {'accuracy','macro avg','weighted avg'}
    rows = []
    for cls in std:
        if cls in skip: continue
        s = std[cls]; t = tta.get(cls, {})
        rows.append({'class': cls, 'support': int(s.get('support',0)),
                     'std_f1': s.get('f1-score',0), 'tta_f1': t.get('f1-score',0),
                     'std_recall': s.get('recall',0), 'tta_recall': t.get('recall',0)})
    df_rep = pd.DataFrame(rows)
    df_rep['f1_gain'] = df_rep['tta_f1'] - df_rep['std_f1']

    print('Top 15 classes most improved by TTA:')
    print(df_rep.sort_values('f1_gain', ascending=False)
               [['class','support','std_f1','tta_f1','f1_gain']].head(15).to_string(index=False))
    print()
    print('Classes still with TTA F1 < 0.5 (scraping candidates):')
    still_bad = df_rep[df_rep['tta_f1'] < 0.5].sort_values('tta_f1')
    print(f'Count: {len(still_bad)}')
    print(still_bad[['class','support','std_f1','tta_f1','f1_gain']].head(20).to_string(index=False))

In [ ]:
if STD_REPORT.exists() and TTA_REPORT.exists():
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    axes[0].hist(df_rep['f1_gain'], bins=40, color=ACCENT2, edgecolor='#0f1117', linewidth=0.4)
    axes[0].axvline(0, color=DANGER, linestyle='--', lw=1.5)
    axes[0].set(xlabel='F1 gain (TTA - Standard)', ylabel='Number of classes',
                title='F1 Gain Distribution from TTA')
    n_imp = (df_rep['f1_gain'] > 0).sum()
    n_wrs = (df_rep['f1_gain'] < 0).sum()
    axes[0].text(0.02, 0.95, f'Improved: {n_imp}\nWorse: {n_wrs}',
                 transform=axes[0].transAxes, fontsize=10, va='top', color='#e0e0e0',
                 bbox=dict(facecolor='#1a1d27', alpha=0.7, edgecolor='#3a3d4d'))
    axes[0].grid(axis='y')

    sc_colors = [ACCENT if v >= 0 else DANGER for v in df_rep['f1_gain']]
    axes[1].scatter(df_rep['support'], df_rep['f1_gain'],
                    c=sc_colors, alpha=0.7, s=40)
    axes[1].axhline(0, color=WARN, linestyle='--', lw=1.5)
    axes[1].set(xlabel='Val support (images)', ylabel='F1 gain from TTA',
                title='TTA Gain vs Class Support\n(rare classes benefit more)')
    axes[1].grid()

    plt.suptitle('TTA Impact — Standard vs TTA (7 passes)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## 10 · Quick Search — Find Any Class by Keyword

In [ ]:
QUERY = 'Tomato'   # change to any keyword

matches = [c for c in all_classes if QUERY.lower() in c.lower()]
print(f"Classes matching '{QUERY}': {len(matches)}\n")
for cls in matches:
    tr, va, te = train_counts.get(cls,0), val_counts.get(cls,0), test_counts.get(cls,0)
    flag = ' 🔴 RARE' if tr < 50 else (' 🟡' if tr < 100 else '')
    print(f'  {cls:<55}  train={tr:>4}  val={va:>3}  test={te:>3}{flag}')